# Unified Poisson Solver Testing
This notebook runs the unified test suites for Problem 1 (Table 1 & 2) as well as the complete NUDFT/NUFFT Accuracy Comparisons, tracking strictly $L_2$ relative errors and computational runtimes.

In [1]:
import os, sys
import pandas as pd
# Main project root
repo_root = r"C:\Users\charl\OneDrive\Documents\GitHub\NUFFTRR_Poisson"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from Tests.CPU.testing_helpers import (
    run_tests_pipeline,
    render_accuracy,
    render_runtime,
    render_table2_accuracy,
    render_table2_runtime
)


# Setup

In [2]:
N_vals_p0 = [32, 64, 128, 256, 512]
M_vals_p0 = [32, 64, 128, 256, 512]
N_fixed_p0 = 32

N_vals_p1 = [32, 64, 128, 256, 512]
M_vals_p1 = [32, 64, 128, 256, 512]
N_fixed_p1 = 32

N_vals_c = [32, 64, 128, 256, 512]
M_vals_c = [32, 64, 128, 256, 512]

MUTE_OUTPUT = True


# Run Problems

In [3]:
METHODS_P0 = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Fixed-jittered-NUDFT", label="Fixed jittered / NUDFT", azu_unif=1, mesh_kind="jittered", solver_azu_unif=1, use_nudft=True),
    dict(name="Fixed-jittered-NUFFT", label="Fixed jittered / NUFFT", azu_unif=1, mesh_kind="jittered", solver_azu_unif=1, use_nudft=False),
]

METHODS_P1 = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Fixed-rand-NUDFT", label="Fixed rand / NUDFT", azu_unif=1, mesh_kind="rand", solver_azu_unif=1, use_nudft=True),
    dict(name="Fixed-rand-NUFFT", label="Fixed rand / NUFFT", azu_unif=1, mesh_kind="rand", solver_azu_unif=1, use_nudft=False),
]

METHODS_COMP = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Unif-NUDFT", label="Uniform / NUDFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=1, use_nudft=True),
    dict(name="Unif-NUFFT", label="Uniform / NUFFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=1, use_nudft=False),
]
for kind in ("rand", "jittered", "clustered", "sine"):
    METHODS_COMP += [
        dict(name=f"Fixed-{kind}-NUDFT", label=f"Fixed {kind} / NUDFT", azu_unif=1, mesh_kind=kind, solver_azu_unif=1, use_nudft=True),
        dict(name=f"Fixed-{kind}-NUFFT", label=f"Fixed {kind} / NUFFT", azu_unif=1, mesh_kind=kind, solver_azu_unif=1, use_nudft=False),
    ]

print("Running Problem 0 - Table 1 (Jittered)...")
df_p0_t1 = run_tests_pipeline(N_vals_p0, M_vals_p0, fixed_other=None, methods=METHODS_P0, test_type="P1_Table1",mute=MUTE_OUTPUT)
print("\nRunning Problem 0 - Table 2 (Jittered)...")
df_p0_t2 = run_tests_pipeline(None, M_vals_p0, fixed_other=N_fixed_p0, methods=METHODS_P0, test_type="P1_Table2",mute=MUTE_OUTPUT)

print("Running Problem 1 - Table 1 (Rand)...")
df_t1 = run_tests_pipeline(N_vals_p1, M_vals_p1, fixed_other=None, methods=METHODS_P1, test_type="P1_Table1",mute=MUTE_OUTPUT)
print("\nRunning Problem 1 - Table 2 (Rand)...")
df_t2 = run_tests_pipeline(None, M_vals_p1, fixed_other=N_fixed_p1, methods=METHODS_P1, test_type="P1_Table2",mute=MUTE_OUTPUT)


print("Running Accuracy Comparisons...")
df_cN = run_tests_pipeline(N_vals_c, None, fixed_other=32, methods=METHODS_COMP, test_type="Accuracy_VaryN", mute=MUTE_OUTPUT)
df_cM = run_tests_pipeline(None, M_vals_c, fixed_other=32, methods=METHODS_COMP, test_type="Accuracy_VaryM", mute=MUTE_OUTPUT)
print("Done.")

Running Problem 0 - Table 1 (Jittered)...

Running Problem 0 - Table 2 (Jittered)...
Running Problem 1 - Table 1 (Rand)...

Running Problem 1 - Table 2 (Rand)...
Running Accuracy Comparisons...
Done.


# Accuracy

In [4]:
print("--- ACCURACY SECTION ---")

print("Problem 0 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same")
render_accuracy(df_p0_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 0 - Table 1")
print("error along rows indicate nudft or nufft is not currently accurate enough")

print("Problem 0 - Table 2 - errors should decrease along rows")
render_table2_accuracy(df_p0_t2, title_prefix=f"Problem 0 - Table 2 (N={N_fixed_p0})")
print("Neumann error for nudft and nufft being constant is an issue. this is either with zero mode or handling fourier coeffs.")

print("Problem 1 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same")
render_accuracy(df_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 1 - Table 1")
print("error along rows indicate nudft or nufft is not currently accurate enough. Inaccuracy is worse than Problem 0, likely due to random mesh being more challenging than jittered.")

print("Problem 1 - Table 2 - errors should decrease along rows")
render_table2_accuracy(df_t2, title_prefix=f"Problem 1 - Table 2 (N={N_fixed_p1})")
print("for nonuniform case dirichlet and neumann errors being constant is concerning. Indicates either mode issue or accuracy issues with algorithm")

print("Problem 2 - Table 1 -All errors should be same along rows, and relatively same along columns")
render_accuracy(df_cN, index_col="N", columns_col="label", title_prefix="Accuracy Comparison (Fixed M=32)")
print("errors for diff meshes getting worse suggests nudft or nufft not accurate")

print("Problem 2 - Table 2 -All errors should decrease along rows, and relatively same along columns")
render_accuracy(df_cM, index_col="M", columns_col="label", title_prefix="Accuracy Comparison (Fixed N=32)")
print("errors for diff meshes getting worse suggests nudft or nufft not accurate")

--- ACCURACY SECTION ---
Problem 0 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same

Problem 0 - Table 1 Accuracy


error along rows indicate nudft or nufft is not currently accurate enough
Problem 0 - Table 2 - errors should decrease along rows

Problem 0 - Table 2 (N=32) Accuracy


Neumann error for nudft and nufft being constant is an issue. this is either with zero mode or handling fourier coeffs.
Problem 1 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same

Problem 1 - Table 1 Accuracy


error along rows indicate nudft or nufft is not currently accurate enough. Inaccuracy is worse than Problem 0, likely due to random mesh being more challenging than jittered.
Problem 1 - Table 2 - errors should decrease along rows

Problem 1 - Table 2 (N=32) Accuracy


for nonuniform case dirichlet and neumann errors being constant is concerning. Indicates either mode issue or accuracy issues with algorithm
Problem 2 - Table 1 -All errors should be same along rows, and relatively same along columns

Accuracy Comparison (Fixed M=32) Accuracy


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
N,,,,,,,,,,,
32,1.10e-05,1.10e-05,1.10e-05,1.25e-05,1.10e-05,1.15e-01,1.10e-05,4.22e-04,1.10e-05,7.98e-05,1.07e-03
64,1.10e-05,1.10e-05,1.10e-05,1.99e-05,1.10e-05,5.47e-02,4.79e-05,1.45e-03,1.10e-05,1.86e-01,1.40e-03
128,1.10e-05,1.10e-05,1.10e-05,1.32e-05,1.10e-05,1.45e-01,2.70e-05,1.90e-03,1.10e-05,2.50e-01,2.79e-04
256,1.10e-05,1.10e-05,1.10e-05,1.12e-05,1.10e-05,1.36e-01,1.10e-05,1.60e-03,1.10e-05,2.62e-01,1.36e-05
512,1.10e-05,1.10e-05,1.10e-05,1.10e-05,1.10e-05,8.64e-02,1.10e-05,1.21e-03,1.10e-05,2.55e-01,1.10e-05


errors for diff meshes getting worse suggests nudft or nufft not accurate
Problem 2 - Table 2 -All errors should decrease along rows, and relatively same along columns

Accuracy Comparison (Fixed N=32) Accuracy


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
M,,,,,,,,,,,
32,1.10e-05,1.10e-05,1.10e-05,1.25e-05,1.10e-05,1.15e-01,1.10e-05,4.22e-04,1.10e-05,7.98e-05,1.07e-03
64,2.66e-06,2.66e-06,2.66e-06,6.64e-06,2.66e-06,1.15e-01,2.66e-06,4.23e-04,2.66e-06,8.06e-05,1.07e-03
128,6.55e-07,6.55e-07,6.55e-07,6.13e-06,6.55e-07,1.15e-01,6.55e-07,4.23e-04,6.55e-07,8.09e-05,1.07e-03
256,1.62e-07,1.62e-07,1.62e-07,6.10e-06,1.62e-07,1.15e-01,1.62e-07,4.23e-04,1.62e-07,8.09e-05,1.07e-03
512,4.04e-08,4.04e-08,4.04e-08,6.10e-06,4.05e-08,1.15e-01,4.05e-08,4.23e-04,4.04e-08,8.09e-05,1.07e-03


errors for diff meshes getting worse suggests nudft or nufft not accurate


# Runtimes

In [5]:
print("--- RUNTIME SECTION ---")
render_runtime(df_p0_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 0 (Jittered) Table 1")
render_table2_runtime(df_p0_t2, title_prefix=f"Problem 0 (Jittered) Table 2 (N={N_fixed_p0})")

render_runtime(df_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 1 (Rand) Table 1")
render_table2_runtime(df_t2, title_prefix=f"Problem 1 (Rand) Table 2 (N={N_fixed_p1})")

render_runtime(df_cN, index_col="N", columns_col="label", title_prefix="Accuracy Comparison (Fixed M=32)")
render_runtime(df_cM, index_col="M", columns_col="label", title_prefix="Accuracy Comparison (Fixed N=32)")

--- RUNTIME SECTION ---

Problem 0 (Jittered) Table 1 Runtime (s)



Problem 0 (Jittered) Table 2 (N=32) Runtime (s)



Problem 1 (Rand) Table 1 Runtime (s)



Problem 1 (Rand) Table 2 (N=32) Runtime (s)



Accuracy Comparison (Fixed M=32) Runtime (s)


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
N,,,,,,,,,,,
32,0.0010,0.0047,0.0360,0.0097,0.0047,0.0033,0.0045,1.3174,0.3596,0.9078,0.4556
64,0.0011,0.0045,0.0348,0.0141,0.0072,0.0057,0.0090,1.7421,0.4574,0.7815,0.5465
128,0.0024,0.0744,0.0494,0.1105,0.0807,0.0831,0.0863,1.6609,0.6794,0.8931,0.7425
256,0.0020,0.2842,0.0530,0.5015,0.3051,0.2935,0.3073,1.3632,0.6467,0.9067,0.8686
512,0.0030,1.1075,0.0532,1.5794,1.1968,1.0948,1.2179,1.0249,0.8331,0.8264,0.7068



Accuracy Comparison (Fixed N=32) Runtime (s)


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
M,,,,,,,,,,,
32,8.48e-04,0.0050,0.0352,0.0086,0.0040,0.0043,0.0087,1.0999,0.3334,0.7871,0.8074
64,0.0020,0.0048,0.0461,0.0109,0.0050,0.0054,0.0109,1.3592,0.5008,1.1923,1.2550
128,0.0033,0.0084,0.0659,0.0203,0.0080,0.0084,0.0186,1.8910,0.7914,2.1528,1.7809
256,0.0063,0.0188,0.1110,0.0385,0.0189,0.0270,0.0336,3.5566,1.2601,2.8354,2.7380
512,0.0096,0.0335,0.1990,0.0751,0.0351,0.0375,0.0739,8.3801,3.2684,5.0995,5.2830
